In [7]:
from pathlib import Path

pasta_base = Path.cwd()
extensoes_imagem = {".avif", ".bmp", ".gif", ".jfif", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
pastas_ignoradas = {".git", ".venv", "venv", "env"}

pastas = [
    pasta_base,
    *(
        pasta
        for pasta in pasta_base.rglob("*")
        if pasta.is_dir()
        and not any(parte in pastas_ignoradas for parte in pasta.relative_to(pasta_base).parts)
        and not any("label" in parte.lower() for parte in pasta.relative_to(pasta_base).parts)
    ),
]
total_imagens = 0

for pasta in sorted(pastas):
    quantidade = sum(
        arquivo.is_file() and arquivo.suffix.lower() in extensoes_imagem
        for arquivo in pasta.iterdir()
    )
    total_imagens += quantidade
    nome_pasta = pasta.relative_to(pasta_base) if pasta != pasta_base else ". (raiz)"
    print(f"{nome_pasta}: {quantidade} imagem(ns)")

print(f"\nTotal de imagens: {total_imagens}")

. (raiz): 0 imagem(ns)
imagens_rotuladas_consolidadas: 114 imagem(ns)
Lixo - Noite: 78 imagem(ns)
plastic: 18 imagem(ns)
trash: 114 imagem(ns)

Total de imagens: 324


In [5]:
pastas_label = sorted(
    pasta
    for pasta in pasta_base.rglob("*")
    if pasta.is_dir()
    and "label" in pasta.name.lower()
    and not any(parte in pastas_ignoradas for parte in pasta.relative_to(pasta_base).parts)
)
total_jsons = 0

for pasta in pastas_label:
    quantidade_jsons = sum(
        arquivo.is_file() and arquivo.suffix.lower() == ".json"
        for arquivo in pasta.iterdir()
    )
    total_jsons += quantidade_jsons
    print(f"{pasta.relative_to(pasta_base)}: {quantidade_jsons} arquivo(s) JSON")

print(f"\nTotal de arquivos JSON em pastas label: {total_jsons}")

imagens_rotuladas_consolidadas\labels: 114 arquivo(s) JSON
Lixo - Noite\labels: 65 arquivo(s) JSON
plastic\label_plastic: 18 arquivo(s) JSON
trash\labels_trash: 114 arquivo(s) JSON

Total de arquivos JSON em pastas label: 311


In [6]:
plano_exclusao = []

for pasta_label in pastas_label:
    nomes_rotulados = {
        arquivo.stem.casefold()
        for arquivo in pasta_label.iterdir()
        if arquivo.is_file() and arquivo.suffix.lower() == ".json"
    }
    pasta_imagens = pasta_label.parent
    imagens_sem_json = [
        arquivo
        for arquivo in pasta_imagens.iterdir()
        if arquivo.is_file()
        and arquivo.suffix.lower() in extensoes_imagem
        and arquivo.stem.casefold() not in nomes_rotulados
    ]
    plano_exclusao.extend(imagens_sem_json)
    print(f"{pasta_imagens.name}: {len(imagens_sem_json)} imagem(ns) sem JSON")

print(f"\nTotal de imagens a excluir: {len(plano_exclusao)}")

for imagem in plano_exclusao:
    imagem.unlink()

print(f"Total de imagens excluídas: {len(plano_exclusao)}")

imagens_rotuladas_consolidadas: 0 imagem(ns) sem JSON
Lixo - Noite: 48 imagem(ns) sem JSON
plastic: 464 imagem(ns) sem JSON
trash: 23 imagem(ns) sem JSON

Total de imagens a excluir: 535
Total de imagens excluídas: 535


In [8]:
import json
import os
import shutil

pasta_dataset = pasta_base / "dataset_model"
pasta_imagens_dataset = pasta_dataset / "images"
pasta_labels_dataset = pasta_dataset / "labels"

if pasta_dataset.exists():
    raise FileExistsError(f"A pasta de destino já existe: {pasta_dataset}")

pares_dataset = []
imagens_referenciadas = set()

for pasta_label in pastas_label:
    pasta_categoria = pasta_label.parent
    for arquivo_json in sorted(pasta_label.glob("*.json")):
        dados_label = json.loads(arquivo_json.read_text(encoding="utf-8"))
        caminho_imagem = dados_label.get("imagePath")
        if not caminho_imagem:
            raise ValueError(f"JSON sem imagePath: {arquivo_json}")

        nome_imagem = Path(caminho_imagem.replace("\\", "/")).name
        imagem_origem = pasta_categoria / nome_imagem
        if not imagem_origem.is_file():
            raise FileNotFoundError(f"Imagem indicada em {arquivo_json.name} não encontrada: {imagem_origem}")
        if imagem_origem.suffix.lower() not in extensoes_imagem:
            raise ValueError(f"Extensão de imagem não suportada: {imagem_origem}")

        chave_imagem = (pasta_categoria.name, imagem_origem.name.casefold())
        if chave_imagem in imagens_referenciadas:
            raise ValueError(f"Imagem referenciada por mais de um JSON: {imagem_origem}")
        imagens_referenciadas.add(chave_imagem)
        pares_dataset.append((pasta_categoria, imagem_origem, arquivo_json))

if not pares_dataset:
    raise RuntimeError("Nenhum par imagem-label foi encontrado.")

for pasta_categoria, imagem_origem, arquivo_json in pares_dataset:
    pasta_imagens_categoria = pasta_imagens_dataset / pasta_categoria.name
    pasta_labels_categoria = pasta_labels_dataset / pasta_categoria.name
    pasta_imagens_categoria.mkdir(parents=True, exist_ok=True)
    pasta_labels_categoria.mkdir(parents=True, exist_ok=True)

    imagem_destino = pasta_imagens_categoria / imagem_origem.name
    label_destino = pasta_labels_categoria / arquivo_json.name
    shutil.copy2(imagem_origem, imagem_destino)

    dados_label = json.loads(arquivo_json.read_text(encoding="utf-8"))
    dados_label["imagePath"] = os.path.relpath(imagem_destino, start=pasta_labels_categoria)
    with label_destino.open("w", encoding="utf-8") as arquivo_saida:
        json.dump(dados_label, arquivo_saida, ensure_ascii=False, indent=2)
        arquivo_saida.write("\n")

imagens_nao_referenciadas = 0
for pasta_label in pastas_label:
    pasta_categoria = pasta_label.parent
    imagens_categoria = [
        arquivo
        for arquivo in pasta_categoria.iterdir()
        if arquivo.is_file() and arquivo.suffix.lower() in extensoes_imagem
    ]
    imagens_nao_referenciadas += sum(
        (pasta_categoria.name, arquivo.name.casefold()) not in imagens_referenciadas
        for arquivo in imagens_categoria
    )

print(f"Pares imagem-label copiados: {len(pares_dataset)}")
print(f"Imagens sem label incluídas: 0 (não referenciadas: {imagens_nao_referenciadas})")
print(f"Destino: {pasta_dataset}")

Pares imagem-label copiados: 311
Imagens sem label incluídas: 0 (não referenciadas: 13)
Destino: c:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 02\dataset_model


In [9]:
from collections import Counter

arquivos_labels_dataset = sorted(pasta_labels_dataset.rglob("*.json"))
contagem_labels_antes = Counter()
total_objetos_rotulados = 0
labels_alteradas = 0

for arquivo_json in arquivos_labels_dataset:
    dados_label = json.loads(arquivo_json.read_text(encoding="utf-8"))
    arquivo_alterado = False

    for shape in dados_label.get("shapes", []):
        label_atual = shape.get("label")
        if label_atual is not None and str(label_atual).strip():
            contagem_labels_antes[str(label_atual)] += 1
            total_objetos_rotulados += 1
            if label_atual != "lixo":
                shape["label"] = "lixo"
                labels_alteradas += 1
                arquivo_alterado = True

    if arquivo_alterado:
        with arquivo_json.open("w", encoding="utf-8") as arquivo_saida:
            json.dump(dados_label, arquivo_saida, ensure_ascii=False, indent=2)
            arquivo_saida.write("\n")

print(f"JSONs verificados: {len(arquivos_labels_dataset)}")
print(f"Objetos rotulados: {total_objetos_rotulados}")
print(f"Labels padronizadas para 'lixo': {labels_alteradas}")
print("Labels encontradas antes da padronização:")
for nome_label, quantidade in sorted(contagem_labels_antes.items()):
    print(f"{nome_label}: {quantidade}")

JSONs verificados: 311
Objetos rotulados: 875
Labels padronizadas para 'lixo': 504
Labels encontradas antes da padronização:
entulho: 16
lixo: 371
lixo_solto: 488
